In [1]:
!pip install ultralytics
!pip install tqdm
!pip install torchmetrics
!pip install pycocotools
!pip install faster-coco-eval

In [2]:
import os
import time
from pathlib import Path
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from torchvision.models.detection import ssd300_vgg16
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [7]:
from pathlib import Path
import torch
import time
from torchvision.models.detection import ssd300_vgg16
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = Path("../../final_unified_dataset")

val_dataset = SSDDataset(root, "val")
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3

model.load_state_dict(torch.load(
    "../../runs/detect/SSDv3_COCO_final/train2/weights/last.pth",
    map_location=device
))

model.to(device)
model.eval()

map_metric = MeanAveragePrecision(
    iou_thresholds=[0.5],
    class_metrics=True,
    backend="faster_coco_eval"
)

inference_times = []
total_tp = 0
total_fp = 0

# --- IoU helper ---
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating SSD"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # --- Timing ---
        if device.type == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            outputs = model(images)
            end.record()

            torch.cuda.synchronize()
            inference_times.append(start.elapsed_time(end))
        else:
            t0 = time.time()
            outputs = model(images)
            t1 = time.time()
            inference_times.append((t1 - t0) * 1000)

        map_metric.update(outputs, targets)

        # --- Precision calculation ---
        for out, tgt in zip(outputs, targets):
            pred_boxes = out["boxes"].cpu()
            gt_boxes = tgt["boxes"].cpu()

            for pb in pred_boxes:
                best_iou = 0

                for gb in gt_boxes:
                    iou = compute_iou(pb, gb)
                    best_iou = max(best_iou, iou)

                if best_iou >= 0.5:
                    total_tp += 1
                else:
                    total_fp += 1

# --- Metrics ---
map_results = map_metric.compute()

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0

ssd_results = {
    "mAP50": map_results["map_50"].item(),
    "mAP50-95": map_results["map"].item(),
    "mean_recall": map_results["mar_100"].item(),
    "precision": precision,
    "inference_ms": sum(inference_times) / len(inference_times),
}

print("\nSSD evaluation results:")
for k, v in ssd_results.items():
    print(f"  {k}: {v:.4f}")

Evaluating SSD: 100%|██████████| 252/252 [01:17<00:00,  3.25it/s]



SSD evaluation results:
  mAP50: 0.2421
  mAP50-95: 0.2421
  mean_recall: 0.3660
  precision: 0.3237
  inference_ms: 279.1620


# If running SSD training locally
###  Google Colab codes are below


In [ ]:
# Run this cell for skipper


root = Path("../../final_unified_dataset")
print("Dataset exists:", root.exists())
print("Train images:", (root / "images/train").exists())
print("Val images:", (root / "images/val").exists())
print("Test images:", (root / "images/test").exists())
print("Train labels:", (root / "labels/train").exists())
print("Val labels:", (root / "labels/val").exists())
print("Test labels:", (root / "labels/test").exists())
print("YAML exists:", (root / "dataset.yaml").exists())
print(Path("../../final_unified_dataset/dataset.yaml").read_text())

# Count images and labels per split
for split in ["train", "val", "test"]:
    img_dir = root / "images" / split
    lbl_dir = root / "labels" / split

    num_images = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    num_labels = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

    print(f"\n{split.capitalize()} split:")
    print(f"  Images: {num_images}")
    print(f"  Labels: {num_labels}")
    


Dataset exists: True
Train images: True
Val images: True
Test images: True
Train labels: True
Val labels: True
Test labels: True
YAML exists: True
path: C:\Users\skipp\Documents\Obsidian Vault\SUTD\Term 6 (Skipper)\Computational Data Science\Proj\test\CDS_2026Spring_project\final_unified_dataset
train: images/train
val:   images/val
test:  images/test

nc: 2
names: ['person', 'bag']


Train split:
  Images: 0
  Labels: 0

Val split:
  Images: 0
  Labels: 0

Test split:
  Images: 0
  Labels: 0


In [39]:
base = "../../final_unified_dataset"

for split in ["train", "val", "test"]:
    label_dir = os.path.join(base, "labels", split)
    counter = Counter()
    for file in os.listdir(label_dir):
        if file.endswith(".txt"):
            with open(os.path.join(label_dir, file)) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        counter[int(parts[0])] += 1
    print(f"{split}: person={counter[0]}, bag={counter[1]}, ratio={counter[1]/counter[0]:.1f}x")
    
    
    

train: person=18649, bag=5488, ratio=0.3x
val: person=2211, bag=660, ratio=0.3x
test: person=2336, bag=698, ratio=0.3x


In [4]:
# SSD model setup (local)
class SSDDataset(Dataset):
    def __init__(self, root, split="train"):
        self.root = Path(root)
        self.img_dir = self.root / f"images/{split}"
        self.label_dir = self.root / f"labels/{split}"

        self.images = list(self.img_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        image = cv2.imread(str(img_path))
        h, w = image.shape[:2]
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.strip().split())

                    x_min = (x - bw/2) * w
                    y_min = (y - bh/2) * h
                    x_max = (x + bw/2) * w
                    y_max = (y + bh/2) * h

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)
                    # +1 because 0 = background in SSD

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        image = torch.tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return image, target

In [ ]:
# SSD Training (Local)
root = "../../final_unified_dataset"

train_dataset = SSDDataset(root, "train")
val_dataset = SSDDataset(root, "val")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                        collate_fn=lambda x: tuple(zip(*x)))

device = torch.device("cpu")
print("Using device:", device)

# Load model
model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# --- Utility functions ---
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2]-box1[0]) * (box1[3]-box1[1])
    box2_area = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def batch_precision_recall(outputs, targets, iou_threshold=0.5):
    correct = 0
    total_pred = 0
    total_gt = 0

    for out, tgt in zip(outputs, targets):
        pred_boxes = out['boxes'].detach().cpu()
        gt_boxes = tgt['boxes'].detach().cpu()

        total_pred += len(pred_boxes)
        total_gt += len(gt_boxes)

        for p in pred_boxes:
            if any(compute_iou(p, g) > iou_threshold for g in gt_boxes):
                correct += 1

    precision = correct / total_pred if total_pred > 0 else 0
    recall = correct / total_gt if total_gt > 0 else 0
    return precision, recall

start_time_total = time.time()
epochs = 20

for epoch in range(epochs):
    epoch_start = time.time()
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"batch_loss": f"{loss.item():.3f}"})

    # --- Validation ---
    model.eval()
    precisions = []
    recalls = []

    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            p, r = batch_precision_recall(outputs, targets)
            precisions.append(p)
            recalls.append(r)

    epoch_precision = sum(precisions) / len(precisions)
    epoch_recall = sum(recalls) / len(recalls)
    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{epochs} | Total Loss: {total_loss:.4f} | "
          f"Precision: {epoch_precision:.3f} | Recall: {epoch_recall:.3f} | "
          f"Epoch Time: {epoch_time:.2f}s")

total_time = time.time() - start_time_total
print(f"\nTotal training time: {total_time/60:.2f} minutes")

In [ ]:
# Saving model 
save_path = "ssd_trained_model_2.pth"

torch.save(model.state_dict(), save_path)

print(f"Model saved to {save_path}")

In [8]:
# Load + evaluate model

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model architecture
model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3  # 2 classes + background

# Load saved weights (local file)
save_path = "../../runs/detect/SSDv3_COCO_final/train2//weights/best.pth"
model.load_state_dict(torch.load(save_path, map_location=device))

# Move to device + eval mode
model.to(device)
model.eval()

print(f"Model loaded from {save_path}")

Model loaded from ../../runs/detect/SSDv3_COCO_final/train2//weights/best.pth


In [10]:
# SSD Inference and Visualization

root = Path("../../final_unified_dataset")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3

save_path = "../../runs/detect/SSDv3_COCO_final/train2//weights/best.pth"
model.load_state_dict(torch.load(save_path, map_location=device))

model.to(device)
model.eval()
print("Loaded SSD model for inference.")

test_image_path = root / "images/train/0a23e1fda36faaab.jpg"

# Read image
img_cv2 = cv2.imread(str(test_image_path))


if img_cv2 is None:
    raise FileNotFoundError(f"Image not found at {test_image_path}")

img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

# Convert to tensor
img_tensor = torch.tensor(img_rgb / 255.0, dtype=torch.float32)\
                .permute(2, 0, 1)\
                .unsqueeze(0)\
                .to(device)

# --- Run inference ---
with torch.no_grad():
    outputs = model(img_tensor)

# --- Draw boxes ---
def draw_boxes(image_path, outputs, conf_threshold=0.25):
    img = Image.open(image_path).copy()
    draw = ImageDraw.Draw(img)

    class_names = {1: "person", 2: "bag"}

    boxes = outputs[0]['boxes'].cpu()
    labels = outputs[0]['labels'].cpu()
    scores = outputs[0]['scores'].cpu()

    for box, label, score in zip(boxes, labels, scores):
        label = int(label)

        if score < conf_threshold:
            continue

        x1, y1, x2, y2 = map(int, box)

        color = "cyan" if label == 1 else "orange"
        name = class_names.get(label, "unknown")

        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1, y1-15), f"{name} {score:.2f}", fill=color)

    return img
img_result = draw_boxes(test_image_path, outputs, conf_threshold=0.25)

plt.figure(figsize=(10, 8))
plt.imshow(img_result)
plt.axis("off")
plt.show()

# --- Ground truth labels ---
label_path = root / "labels/train/0a23e1fda36faaab.txt"

if label_path.exists():
    print(f"\nGround-truth annotations from {label_path}:")
    with open(label_path) as f:
        for i, line in enumerate(f.readlines()):
            cls, cx, cy, bw, bh = map(float, line.strip().split())
            cls_name = "person" if int(cls) == 0 else "bag"
            print(f"{i+1}. Class: {cls_name}, Center: ({cx:.3f},{cy:.3f}), Size: {bw:.3f}x{bh:.3f}")
else:
    print("No ground-truth labels found for this image.")

Loaded SSD model for inference.


FileNotFoundError: Image not found at ..\..\final_unified_dataset\images\train\0a23e1fda36faaab.jpg

In [22]:
from pathlib import Path
import torch
import time
from torchvision.models.detection import ssd300_vgg16
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = Path("../../final_unified_dataset")

val_dataset = SSDDataset(root, "val")
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3

model.load_state_dict(torch.load(
    "../../runs/detect/SSDv3_COCO_final/train2/weights/best.pth",
    map_location=device
))

model.to(device)
model.eval()

map_metric = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)
inference_times = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating SSD"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        if device.type == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            outputs = model(images)
            end.record()

            torch.cuda.synchronize()
            inference_times.append(start.elapsed_time(end))

        else:
            t0 = time.time()
            outputs = model(images)
            t1 = time.time()
            inference_times.append((t1 - t0) * 1000)

        map_metric.update(outputs, targets)

map_results = map_metric.compute()

ssd_results = {
    "mAP50": map_results["map_50"].item(),
    "mAP50-95": map_results["map"].item(),
    "mean_recall": map_results["mar_100"].item(),
    "inference_ms": sum(inference_times) / len(inference_times),
}

print("\nSSD evaluation results:")
for k, v in ssd_results.items():
    print(f"  {k}: {v:.4f}")

ModuleNotFoundError: `MAP` metric requires that `pycocotools` or `faster-coco-eval` installed. Please install with `pip install pycocotools` or `pip install faster-coco-eval` or `pip install torchmetrics[detection]`.

# Google Colab SSD Training


In [ ]:
# Google Colab - 01
from pathlib import Path
from google.colab import drive
import zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/Colab Notebooks/final_unified_dataset.zip"

extract_root = Path("/content")
dataset_path = extract_root / "final_unified_dataset"
if not dataset_path.exists():
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print(f"Extracted dataset to {extract_root}")
else:
    print(f"Dataset already extracted at {extract_root}")

root = extract_root

print("\nDataset summary:")
print("Train images folder exists:", (root / "final_unified_dataset/images/train").exists())
print("Val images folder exists:", (root / "final_unified_dataset/images/val").exists())
print("Test images folder exists:", (root / "final_unified_dataset/images/test").exists())
print("Train labels folder exists:", (root / "final_unified_dataset/labels/train").exists())
print("Val labels folder exists:", (root / "final_unified_dataset/labels/val").exists())
print("Test labels folder exists:", (root / "final_unified_dataset/labels/test").exists())
print("YAML exists:", (root / "final_unified_dataset/dataset.yaml").exists())

for split in ["train", "val", "test"]:
    img_dir = root / "final_unified_dataset/images" / split
    lbl_dir = root / "final_unified_dataset/labels" / split

    num_images = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    num_labels = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

    print(f"\n{split.capitalize()} split:")
    print(f"  Images: {num_images}")
    print(f"  Labels: {num_labels}")

In [ ]:
# Google Colab - 02 - SSD model setup
class SSDDataset(Dataset):
    def __init__(self, root, split="train"):
        self.root = Path(root)
        self.img_dir = self.root / "images" / split
        self.label_dir = self.root / "labels" / split

        if not self.img_dir.exists():
            raise FileNotFoundError(f"Image directory {self.img_dir} not found")
        if not self.label_dir.exists():
            raise FileNotFoundError(f"Label directory {self.label_dir} not found")

        self.images = sorted(list(self.img_dir.glob("*.jpg")) + list(self.img_dir.glob("*.png")))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / f"{img_path.stem}.txt"
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Could not read image {img_path}")
        h, w = image.shape[:2]
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        boxes = []
        labels = []
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.strip().split())
                    # Convert YOLO normalized format to pixel coordinates
                    x_min = (x - bw / 2) * w
                    y_min = (y - bh / 2) * h
                    x_max = (x + bw / 2) * w
                    y_max = (y + bh / 2) * h
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)  # +1 for SSD background class

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        # Convert image to tensor [C,H,W] and normalize
        image = torch.tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return image, target

In [26]:
# Google Colab - 03 - SSD Training (Colab) with progress bar + precision/recall
import time
import torch
from torchvision.models.detection import ssd300_vgg16
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Colab dataset path
root = Path("/content/final_unified_dataset")

train_dataset = SSDDataset(root, "train")
val_dataset = SSDDataset(root, "val")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                        collate_fn=lambda x: tuple(zip(*x)))

# GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3  # 2 classes + background
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2]-box1[0]) * (box1[3]-box1[1])
    box2_area = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def batch_precision_recall(outputs, targets, iou_threshold=0.5):
    correct = 0
    total_pred = 0
    total_gt = 0

    for out, tgt in zip(outputs, targets):
        pred_boxes = out['boxes'].detach().cpu()
        gt_boxes = tgt['boxes'].detach().cpu()
        total_pred += len(pred_boxes)
        total_gt += len(gt_boxes)

        for p in pred_boxes:
            if any(compute_iou(p, g) > iou_threshold for g in gt_boxes):
                correct += 1

    precision = correct / total_pred if total_pred > 0 else 0
    recall = correct / total_gt if total_gt > 0 else 0
    return precision, recall

# Training loop 
start_time_total = time.time()
epochs = 50

for epoch in range(epochs):
    epoch_start = time.time()
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"batch_loss": f"{loss.item():.3f}"})

    model.eval()
    precisions = []
    recalls = []

    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            p, r = batch_precision_recall(outputs, targets)
            precisions.append(p)
            recalls.append(r)

    epoch_precision = sum(precisions) / len(precisions)
    epoch_recall = sum(recalls) / len(recalls)
    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{epochs} | Total Loss: {total_loss:.4f} | "
          f"Precision: {epoch_precision:.3f} | Recall: {epoch_recall:.3f} | "
          f"Epoch Time: {epoch_time:.2f}s")

total_time = time.time() - start_time_total
print(f"\nTotal training time: {total_time/60:.2f} minutes")

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [27]:
print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))

Train dataset size: 0
Val dataset size: 0


# Results from most recent testing 
 <div style="
    max-height: 300px;
    overflow-y: scroll;
    border: 1px solid #ccc;
    padding: 10px;
">


Using device: cuda
Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth

100%|██████████| 136M/136M [00:01<00:00, 108MB/s]
Epoch 1/50: 100%|██████████| 2186/2186 [04:04<00:00,  8.95it/s, batch_loss=3.170]

Epoch 1/50 | Total Loss: 7236.6588 | Precision: 0.021 | Recall: 1.648 | Epoch Time: 314.33s

Epoch 2/50: 100%|██████████| 2186/2186 [04:09<00:00,  8.75it/s, batch_loss=4.978]

Epoch 2/50 | Total Loss: 6134.4878 | Precision: 0.024 | Recall: 1.576 | Epoch Time: 321.24s

Epoch 3/50: 100%|██████████| 2186/2186 [04:14<00:00,  8.59it/s, batch_loss=0.352]

Epoch 3/50 | Total Loss: 5531.4034 | Precision: 0.067 | Recall: 1.429 | Epoch Time: 323.14s

Epoch 4/50: 100%|██████████| 2186/2186 [04:14<00:00,  8.60it/s, batch_loss=1.161]

Epoch 4/50 | Total Loss: 5007.6735 | Precision: 0.054 | Recall: 1.463 | Epoch Time: 324.33s

Epoch 5/50: 100%|██████████| 2186/2186 [04:15<00:00,  8.57it/s, batch_loss=1.862]

Epoch 5/50 | Total Loss: 4461.9890 | Precision: 0.131 | Recall: 1.501 | Epoch Time: 320.75s

Epoch 6/50: 100%|██████████| 2186/2186 [04:14<00:00,  8.57it/s, batch_loss=2.139]

Epoch 6/50 | Total Loss: 4107.2790 | Precision: 0.194 | Recall: 1.474 | Epoch Time: 317.54s

Epoch 7/50: 100%|██████████| 2186/2186 [04:15<00:00,  8.57it/s, batch_loss=0.915]

Epoch 7/50 | Total Loss: 3757.5922 | Precision: 0.315 | Recall: 1.218 | Epoch Time: 309.46s

Epoch 8/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.65it/s, batch_loss=1.739]

Epoch 8/50 | Total Loss: 3556.3654 | Precision: 0.364 | Recall: 1.322 | Epoch Time: 303.25s

Epoch 9/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.64it/s, batch_loss=0.451]

Epoch 9/50 | Total Loss: 3300.5416 | Precision: 0.317 | Recall: 1.263 | Epoch Time: 306.65s

Epoch 10/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.64it/s, batch_loss=0.987]

Epoch 10/50 | Total Loss: 3079.8132 | Precision: 0.390 | Recall: 1.119 | Epoch Time: 304.45s

Epoch 11/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.63it/s, batch_loss=2.674]

Epoch 11/50 | Total Loss: 2953.5354 | Precision: 0.350 | Recall: 1.139 | Epoch Time: 303.25s

Epoch 12/50: 100%|██████████| 2186/2186 [04:14<00:00,  8.60it/s, batch_loss=0.223]

Epoch 12/50 | Total Loss: 2791.4169 | Precision: 0.400 | Recall: 1.012 | Epoch Time: 304.96s

Epoch 13/50: 100%|██████████| 2186/2186 [04:14<00:00,  8.60it/s, batch_loss=2.313]

Epoch 13/50 | Total Loss: 2755.9244 | Precision: 0.445 | Recall: 1.061 | Epoch Time: 302.47s

Epoch 14/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.66it/s, batch_loss=3.157]

Epoch 14/50 | Total Loss: 2548.9235 | Precision: 0.439 | Recall: 0.952 | Epoch Time: 300.73s

Epoch 15/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=1.329]

Epoch 15/50 | Total Loss: 2462.6414 | Precision: 0.482 | Recall: 1.048 | Epoch Time: 301.20s

Epoch 16/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=3.905]

Epoch 16/50 | Total Loss: 2475.0559 | Precision: 0.485 | Recall: 1.041 | Epoch Time: 301.14s

Epoch 17/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=0.184]

Epoch 17/50 | Total Loss: 2257.1844 | Precision: 0.526 | Recall: 0.993 | Epoch Time: 299.71s

Epoch 18/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.66it/s, batch_loss=2.304]

Epoch 18/50 | Total Loss: 2297.9195 | Precision: 0.550 | Recall: 0.985 | Epoch Time: 300.55s

Epoch 19/50: 100%|██████████| 2186/2186 [04:14<00:00,  8.60it/s, batch_loss=0.514]

Epoch 19/50 | Total Loss: 2141.3947 | Precision: 0.556 | Recall: 0.891 | Epoch Time: 302.28s

Epoch 20/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=1.393]

Epoch 20/50 | Total Loss: 2184.1889 | Precision: 0.562 | Recall: 0.923 | Epoch Time: 300.12s

Epoch 21/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.63it/s, batch_loss=1.049]

Epoch 21/50 | Total Loss: 2045.5751 | Precision: 0.596 | Recall: 0.913 | Epoch Time: 299.67s

Epoch 22/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.64it/s, batch_loss=2.049]

Epoch 22/50 | Total Loss: 2069.6211 | Precision: 0.559 | Recall: 0.885 | Epoch Time: 300.96s

Epoch 23/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.63it/s, batch_loss=0.882]

Epoch 23/50 | Total Loss: 1954.4370 | Precision: 0.575 | Recall: 0.882 | Epoch Time: 301.57s

Epoch 24/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.62it/s, batch_loss=1.068]

Epoch 24/50 | Total Loss: 2025.5290 | Precision: 0.540 | Recall: 0.969 | Epoch Time: 300.42s

Epoch 25/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.64it/s, batch_loss=1.871]

Epoch 25/50 | Total Loss: 1894.2323 | Precision: 0.523 | Recall: 0.891 | Epoch Time: 300.84s

Epoch 26/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.67it/s, batch_loss=0.232]

Epoch 26/50 | Total Loss: 1897.0612 | Precision: 0.605 | Recall: 0.828 | Epoch Time: 298.52s

Epoch 27/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.64it/s, batch_loss=0.329]

Epoch 27/50 | Total Loss: 1897.3285 | Precision: 0.563 | Recall: 0.886 | Epoch Time: 299.42s

Epoch 28/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.67it/s, batch_loss=0.456]

Epoch 28/50 | Total Loss: 1768.6445 | Precision: 0.588 | Recall: 0.820 | Epoch Time: 299.08s

Epoch 29/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.70it/s, batch_loss=0.213]

Epoch 29/50 | Total Loss: 1772.1453 | Precision: 0.549 | Recall: 0.903 | Epoch Time: 297.34s

Epoch 30/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.69it/s, batch_loss=2.083]

Epoch 30/50 | Total Loss: 1741.0102 | Precision: 0.560 | Recall: 0.838 | Epoch Time: 298.20s

Epoch 31/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=2.037]

Epoch 31/50 | Total Loss: 1811.1904 | Precision: 0.588 | Recall: 0.887 | Epoch Time: 300.22s

Epoch 32/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.65it/s, batch_loss=0.765]

Epoch 32/50 | Total Loss: 1683.7821 | Precision: 0.659 | Recall: 0.853 | Epoch Time: 299.08s

Epoch 33/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.66it/s, batch_loss=0.058]

Epoch 33/50 | Total Loss: 1966.4096 | Precision: 0.553 | Recall: 0.867 | Epoch Time: 300.09s

Epoch 34/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=0.816]

Epoch 34/50 | Total Loss: 1639.8946 | Precision: 0.627 | Recall: 0.822 | Epoch Time: 300.08s

Epoch 35/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.63it/s, batch_loss=2.744]

Epoch 35/50 | Total Loss: 1587.4112 | Precision: 0.607 | Recall: 0.855 | Epoch Time: 299.84s

Epoch 36/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.62it/s, batch_loss=0.393]

Epoch 36/50 | Total Loss: 1798.9846 | Precision: 0.649 | Recall: 0.874 | Epoch Time: 300.79s

Epoch 37/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.66it/s, batch_loss=3.444]

Epoch 37/50 | Total Loss: 1544.0557 | Precision: 0.674 | Recall: 0.840 | Epoch Time: 298.81s

Epoch 38/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.67it/s, batch_loss=0.793]

Epoch 38/50 | Total Loss: 1541.1338 | Precision: 0.622 | Recall: 0.839 | Epoch Time: 298.09s

Epoch 39/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.68it/s, batch_loss=0.172]

Epoch 39/50 | Total Loss: 1699.0426 | Precision: 0.620 | Recall: 0.839 | Epoch Time: 298.36s

Epoch 40/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.69it/s, batch_loss=0.314]

Epoch 40/50 | Total Loss: 1527.3974 | Precision: 0.641 | Recall: 0.859 | Epoch Time: 297.63s

Epoch 41/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.70it/s, batch_loss=0.670]

Epoch 41/50 | Total Loss: 1522.9978 | Precision: 0.629 | Recall: 0.808 | Epoch Time: 297.98s

Epoch 42/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.70it/s, batch_loss=1.109]

Epoch 42/50 | Total Loss: 1543.8154 | Precision: 0.624 | Recall: 0.801 | Epoch Time: 297.28s

Epoch 43/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.70it/s, batch_loss=1.740]

Epoch 43/50 | Total Loss: 1591.5283 | Precision: 0.637 | Recall: 0.822 | Epoch Time: 297.57s

Epoch 44/50: 100%|██████████| 2186/2186 [04:11<00:00,  8.68it/s, batch_loss=1.021]

Epoch 44/50 | Total Loss: 1446.5674 | Precision: 0.679 | Recall: 0.806 | Epoch Time: 299.57s

Epoch 45/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=6.224]

Epoch 45/50 | Total Loss: 1547.4737 | Precision: 0.659 | Recall: 0.824 | Epoch Time: 299.00s

Epoch 46/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.65it/s, batch_loss=1.599]

Epoch 46/50 | Total Loss: 1426.8862 | Precision: 0.664 | Recall: 0.892 | Epoch Time: 299.72s

Epoch 47/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.65it/s, batch_loss=0.596]

Epoch 47/50 | Total Loss: 1510.7163 | Precision: 0.593 | Recall: 0.790 | Epoch Time: 299.61s

Epoch 48/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.63it/s, batch_loss=0.195]

Epoch 48/50 | Total Loss: 1464.2120 | Precision: 0.671 | Recall: 0.846 | Epoch Time: 300.37s

Epoch 49/50: 100%|██████████| 2186/2186 [04:13<00:00,  8.64it/s, batch_loss=0.143]

Epoch 49/50 | Total Loss: 1356.4235 | Precision: 0.656 | Recall: 0.844 | Epoch Time: 299.78s

Epoch 50/50: 100%|██████████| 2186/2186 [04:12<00:00,  8.64it/s, batch_loss=0.457]

Epoch 50/50 | Total Loss: 1448.3441 | Precision: 0.645 | Recall: 0.823 | Epoch Time: 299.08s

Total training time: 252.34 minutes
</div>

In [ ]:
# Google Colab - 04 - Save trained SSD model in colab and to google drive
import torch
from pathlib import Path
from google.colab import drive

# Mount Drive (if not already mounted)
drive.mount('/content/drive')

# Local save (Colab VM)
local_path = "/content/final_unified_dataset/ssd_trained_model_2.pth"
torch.save(model.state_dict(), local_path)
print(f"Model saved locally to {local_path}")

# Google Drive save
drive_dir = Path("/content/drive/MyDrive/Colab Notebooks")
drive_dir.mkdir(parents=True, exist_ok=True)

drive_path = drive_dir / "ssd_trained_model_2.pth"
torch.save(model.state_dict(), drive_path)

print(f"Model saved to Google Drive at {drive_path}")

In [ ]:
# Google Colab - 05 - Load the model weights and evaluating model
model = ssd300_vgg16(weights="DEFAULT")
save_path = "/content/final_unified_dataset/ssd_trained_model_2.pth"
model.head.classification_head.num_classes = 3
model.load_state_dict(torch.load(save_path))
model.to(device)
model.eval()  # switch to evaluation mode

In [ ]:
# Google Colab - 06 - SSD Inference and Visualization
# --- Load trained model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3  # 2 classes + background

# Load your trained weights
save_path = "/content/final_unified_dataset/ssd_trained_model_2.pth"
model.load_state_dict(torch.load(save_path, map_location=device))
model.to(device)
model.eval()
print("Loaded SSD model for inference.")

# --- Select test image ---
test_image_path = "/content/final_unified_dataset/images/train/0a23e1fda36faaab.jpg"
img_cv2 = cv2.imread(str(test_image_path))  # Convert Path to string
img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)
img_tensor = torch.tensor(img_rgb / 255.0, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

# --- Run inference ---
with torch.no_grad():
    outputs = model(img_tensor)

# --- Draw boxes ---
def draw_boxes(image_path, outputs, conf_threshold=0.25):
    img = Image.open(image_path).copy()
    draw = ImageDraw.Draw(img)
    class_names = {1: "person", 2: "bag"}  # SSD +1 offset

    boxes = outputs[0]['boxes'].cpu()
    labels = outputs[0]['labels'].cpu()
    scores = outputs[0]['scores'].cpu()

    for box, label, score in zip(boxes, labels, scores):
        label = int(label)  # <-- convert tensor to int
        if score < conf_threshold:
            continue
        x1, y1, x2, y2 = map(int, box)
        color = "cyan" if label == 1 else "orange"
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1, y1-15), f"{class_names[label]} {score:.2f}", fill=color)
    return img

# --- Visualize ---
img_result = draw_boxes(test_image_path, outputs, conf_threshold=0.25)
plt.figure(figsize=(10,8))
plt.imshow(img_result)
plt.axis("off")
plt.show()

# --- Optional: Print ground-truth annotations ---
label_path = Path("/content/final_unified_dataset/labels/test/example.txt")
if label_path.exists():
    print(f"\nGround-truth annotations from {label_path}:")
    with open(label_path) as f:
        for i, line in enumerate(f.readlines()):
            cls, cx, cy, bw, bh = map(float, line.strip().split())
            cls_name = "person" if int(cls)==0 else "bag"
            print(f"{i+1}. Class: {cls_name}, Center: ({cx:.3f},{cy:.3f}), Size: {bw:.3f}x{bh:.3f}")
else:
    print("No ground-truth labels found for this image.")

In [ ]:
# Google Colab - 07 - Testing model against validation images

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
root = "/content/final_unified_dataset"
val_dataset = SSDDataset(root, "val")
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                        collate_fn=lambda x: tuple(zip(*x)))

# Load model
model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3  # 2 classes + background
model.load_state_dict(torch.load("/content/final_unified_dataset/ssd_trained_model_2.pth", map_location=device))
model.to(device)
model.eval()

# TorchMetrics
map_metric = MeanAveragePrecision(iou_thresholds=[0.5, 0.5], class_metrics=True)
inference_times = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating SSD"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Measure inference time
        start = torch.cuda.Event(enable_timing=True) if device.type=='cuda' else None
        end = torch.cuda.Event(enable_timing=True) if device.type=='cuda' else None
        if device.type=='cuda':
            start.record()

        outputs = model(images)

        if device.type=='cuda':
            end.record()
            torch.cuda.synchronize()
            inference_times.append(start.elapsed_time(end))
        else:
            import time
            t0 = time.time()
            outputs = model(images)
            t1 = time.time()
            inference_times.append((t1-t0)*1000)  # ms

        # Update metric
        map_metric.update(outputs, targets)

# Compute final metrics
map_results = map_metric.compute()

ssd_results = {
    "mAP50": map_results['map_50'].item(),
    "mAP50-95": map_results['map'].item(),
    "mean_recall": map_results['mar_100'].item(),  # max recalls averaged
    "inference_ms": sum(inference_times)/len(inference_times),
}

print("SSD evaluation results:")
for k, v in ssd_results.items():
    print(f"  {k}: {v:.4f}")


# Most recent evaluation 
Evaluating SSD: 100%|██████████| 586/586 [00:48<00:00, 12.04it/s]

SSD evaluation results:
  mAP50: 0.5731
  mAP50-95: 0.5731
  mean_recall: 0.7003
  inference_ms: 66.4766


# Webcam Script

In [ ]:
from collections import deque
import cv2
import torch
from torchvision.models.detection import ssd300_vgg16

# ── Load SSD model ────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3
model.load_state_dict(torch.load("./ssd_trained_model_2.pth", map_location=device))
model.to(device)
model.eval()

# ── SSD Inference ─────────────────────────────────────────────────────────────

def ssd_predict(frame, conf_threshold=0.3):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    img_tensor = torch.tensor(img_rgb / 255.0, dtype=torch.float32)\
        .permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)[0]

    boxes = outputs["boxes"].cpu()
    labels = outputs["labels"].cpu()
    scores = outputs["scores"].cpu()

    people = []
    bags = []

    for box, label, score in zip(boxes, labels, scores):
        if score < conf_threshold:
            continue

        det = {
            "box": box.tolist(),
            "conf": float(score)
        }

        if int(label) == 1:
            people.append(det)
        elif int(label) == 2:
            bags.append(det)

    return people, bags

# ── Drawing ───────────────────────────────────────────────────────────────────

def draw_detections(frame, people, bags, stable_bags):
    # Draw people
    for p in people:
        x1, y1, x2, y2 = map(int, p["box"])
        conf = p["conf"]

        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
        cv2.putText(frame, f"person {conf:.2f}",
                    (x1, max(y1-8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    # Draw bags
    for b in bags:
        x1, y1, x2, y2 = map(int, b["box"])
        conf = b["conf"]

        color = (0, 0, 255) if stable_bags else (0, 165, 255)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"bag {conf:.2f}",
                    (x1, max(y1-8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Stats
    cv2.putText(frame,
                f"People: {len(people)}  Bags: {len(bags)}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Alert
    if stable_bags:
        cv2.putText(frame, "!! BAG DETECTED !!",
                    (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

    return frame

# ── Webcam loop ───────────────────────────────────────────────────────────────

cap = cv2.VideoCapture("../../IMG_2162.mp4")

if not cap.isOpened():
    print("Error: Could not open webcam")
    exit()

# Temporal smoothing (reduces flickering)
bag_history = deque(maxlen=5)

print("Running SSD only... press 'q' to quit")
frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # frame = cv2.resize(frame, (300, 300)) # Resizes window to smaller size for faster processing (optional)
    people, bags = ssd_predict(frame, conf_threshold=0.4)
    
    # Speeds up video
    # % 2 → 2x faster
    # % 3 → 3x faster
    # % 5 → 5x faster
    # frame_count += 1
    # if frame_count % 2 != 0:
    #     continue
        
        

    # Temporal filtering
    bag_history.append(len(bags) > 0)
    stable_bags = sum(bag_history) >= 3

    annotated = draw_detections(frame.copy(), people, bags, stable_bags)

    cv2.imshow("SSD Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()